# Variant 3 — Advanced (CrossAttn + GraphReasoning + ClauseHierarchy)
**DistilBERT → token-level gating + clause cross-attention + graph reasoning + clause attention pooling → fusion → ClassHead × 2.**

In [1]:
!pip install transformers scikit-learn pandas torch tqdm --quiet
print("Libraries installed.")

Libraries installed.


In [2]:
import os

DATA_DIR   = "/kaggle/input/datasets/ervarishitha11/datasets-inlp"
OUTPUT_DIR = "/kaggle/working"

files_needed = [
    "QAEvasion.csv",
    "raw_train_biden.csv","raw_val_biden.csv","raw_test_biden.csv",
    "raw_train_trump.csv","raw_val_trump.csv","raw_test_trump.csv",
    "raw_train_bernie.csv","raw_val_bernie.csv","raw_test_bernie.csv",
]
print("Checking files:")
all_ok = True
for f in files_needed:
    ok = os.path.exists(os.path.join(DATA_DIR, f))
    print(f"  {'OK' if ok else 'MISSING':7s}  {f}")
    if not ok: all_ok = False
print("All files found!" if all_ok else "Fix missing files.")

Checking files:
  OK       QAEvasion.csv
  OK       raw_train_biden.csv
  OK       raw_val_biden.csv
  OK       raw_test_biden.csv
  OK       raw_train_trump.csv
  OK       raw_val_trump.csv
  OK       raw_test_trump.csv
  OK       raw_train_bernie.csv
  OK       raw_val_bernie.csv
  OK       raw_test_bernie.csv
All files found!


In [3]:
import os, re, random, warnings, itertools
import pandas as pd, numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type=="cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

BERT_MODEL     = "distilbert-base-uncased"
MAX_LEN        = 256
STANCE_MAX_LEN = 128
Q_MAX_LEN      = 64
MAX_CLAUSES    = 6
CLAUSE_LEN     = 64
BATCH_SIZE     = 16    # smaller: clause encoding multiplies memory
EPOCHS         = 5
DROPOUT        = 0.3
WARMUP_RATIO   = 0.1
FREEZE_EPOCHS  = 2
LR_EVASION     = 1e-5
LR_STANCE      = 2e-5

VARIANT_NAME    = "Variant 3 — Advanced (CrossAttn + GraphReasoning + ClauseHierarchy)"
SAVE_TAG        = "v3"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_v3_advanced.pt")

EVASION_LABELS   = ["Non-Evasive", "Partially Evasive", "Evasive"]
EVASION_LABEL2ID = {l: i for i, l in enumerate(EVASION_LABELS)}
STANCE_LABELS    = ["FAVOR", "AGAINST"]
STANCE_LABEL2ID  = {l: i for i, l in enumerate(STANCE_LABELS)}

EVASION_CSV       = f"{DATA_DIR}/QAEvasion.csv"
STANCE_TRAIN_CSVS = [f"{DATA_DIR}/raw_train_biden.csv",f"{DATA_DIR}/raw_train_trump.csv",f"{DATA_DIR}/raw_train_bernie.csv"]
STANCE_VAL_CSVS   = [f"{DATA_DIR}/raw_val_biden.csv",  f"{DATA_DIR}/raw_val_trump.csv",  f"{DATA_DIR}/raw_val_bernie.csv"]
STANCE_TEST_CSVS  = [f"{DATA_DIR}/raw_test_biden.csv", f"{DATA_DIR}/raw_test_trump.csv", f"{DATA_DIR}/raw_test_bernie.csv"]

print(f"  Epochs:{EPOCHS}  BERT frozen first {FREEZE_EPOCHS} epochs")
print(f"  LR evasion:{LR_EVASION}  LR stance:{LR_STANCE}  Batch:{BATCH_SIZE}")
print(f"  {VARIANT_NAME}")

Device: cuda
GPU: Tesla T4
  Epochs:5  BERT frozen first 2 epochs
  LR evasion:1e-05  LR stance:2e-05  Batch:16
  Variant 3 — Advanced (CrossAttn + GraphReasoning + ClauseHierarchy)


In [4]:
def map_evasion_label(raw):
    raw = str(raw).strip()
    if raw.startswith("1."): return "Non-Evasive"
    elif raw == "2.3 Partial/half-answer": return "Partially Evasive"
    else: return "Evasive"

def load_evasion(filepath):
    print("Loading QA Evasion...")
    df = pd.read_csv(filepath)
    df["question"] = df["interview_question"].fillna("").str.strip()
    df["answer"]   = df["interview_answer"].fillna("").str.strip()
    df["coarse_label"] = df["label"].apply(map_evasion_label)
    df["label_id"]     = df["coarse_label"].map(EVASION_LABEL2ID).astype(int)
    df = df.dropna(subset=["question","answer","label_id"]).reset_index(drop=True)
    print(f"  Total: {len(df)}")
    for lbl in EVASION_LABELS:
        n = (df["coarse_label"]==lbl).sum()
        print(f"  {lbl:22s}: {n:4d} ({n/len(df)*100:.1f}%)")
    idx = list(range(len(df)))
    lbl = df["label_id"].tolist()
    itr,itmp,_,ytmp = train_test_split(idx,lbl,test_size=0.20,random_state=SEED,stratify=lbl)
    iva,ite,_,_     = train_test_split(itmp,ytmp,test_size=0.50,random_state=SEED,stratify=ytmp)
    q,a,l = df["question"].tolist(),df["answer"].tolist(),df["label_id"].tolist()
    def ex(ii): return [q[i] for i in ii],[a[i] for i in ii],[l[i] for i in ii]
    tr,va,te = ex(itr),ex(iva),ex(ite)
    print(f"  Train:{len(itr)}  Val:{len(iva)}  Test:{len(ite)}")
    return tr,va,te

def load_stance_split(filepaths, split_name):
    print(f"Loading Stance [{split_name}]...")
    dfs=[]
    for fp in filepaths:
        if os.path.exists(fp):
            d=pd.read_csv(fp); dfs.append(d)
            print(f"  {os.path.basename(fp):30s}  {len(d)} rows")
    df=pd.concat(dfs,ignore_index=True)
    df=df[df["Stance"].isin(STANCE_LABEL2ID)].reset_index(drop=True)
    df["label_id"]=df["Stance"].map(STANCE_LABEL2ID).astype(int)
    df["target"]=df["Target"].fillna("").str.strip()
    df["tweet"]=df["Tweet"].fillna("").str.strip()
    df=df.dropna(subset=["target","tweet","label_id"]).reset_index(drop=True)
    print(f"  Total: {len(df)}")
    return df["target"].tolist(),df["tweet"].tolist(),df["label_id"].tolist()

print("="*60)
(Qetr,Aetr,yetr),(Qeva,Aeva,yeva),(Qete,Aete,yete)=load_evasion(EVASION_CSV)
print()
Qstr,Astr,ystr=load_stance_split(STANCE_TRAIN_CSVS,"train")
print()
Qsva,Asva,ysva=load_stance_split(STANCE_VAL_CSVS,"val")
print()
Qste,Aste,yste=load_stance_split(STANCE_TEST_CSVS,"test")
print("="*60)
print("Datasets loaded.")

Loading QA Evasion...
  Total: 3448
  Non-Evasive           : 1540 (44.7%)
  Partially Evasive     :   79 (2.3%)
  Evasive               : 1829 (53.0%)
  Train:2758  Val:345  Test:345

Loading Stance [train]...
  raw_train_biden.csv             5806 rows
  raw_train_trump.csv             6362 rows
  raw_train_bernie.csv            5056 rows
  Total: 17224

Loading Stance [val]...
  raw_val_biden.csv               745 rows
  raw_val_trump.csv               814 rows
  raw_val_bernie.csv              634 rows
  Total: 2193

Loading Stance [test]...
  raw_test_biden.csv              745 rows
  raw_test_trump.csv              777 rows
  raw_test_bernie.csv             635 rows
  Total: 2157
Datasets loaded.


In [5]:
def save_split(q,a,l,lmap,fname):
    pd.DataFrame({"question":q,"answer":a,"label_id":l,
                  "label":[lmap[x] for x in l]
    }).to_csv(os.path.join(OUTPUT_DIR,fname),index=False)
    print(f"  Saved {fname}: {len(l)} rows")

save_split(Qetr,Aetr,yetr,EVASION_LABELS,"evasion_train.csv")
save_split(Qeva,Aeva,yeva,EVASION_LABELS,"evasion_val.csv")
save_split(Qete,Aete,yete,EVASION_LABELS,"evasion_test.csv")
save_split(Qstr,Astr,ystr,STANCE_LABELS,"stance_train.csv")
save_split(Qsva,Asva,ysva,STANCE_LABELS,"stance_val.csv")
save_split(Qste,Aste,yste,STANCE_LABELS,"stance_test.csv")
print("Splits saved.")

  Saved evasion_train.csv: 2758 rows
  Saved evasion_val.csv: 345 rows
  Saved evasion_test.csv: 345 rows
  Saved stance_train.csv: 17224 rows
  Saved stance_val.csv: 2193 rows
  Saved stance_test.csv: 2157 rows
Splits saved.


In [6]:
tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL)

def split_into_clauses(text):
    """Split answer/tweet into clauses on punctuation and discourse connectives."""
    parts = re.split(
        r'(?<=[.!?])\s+|(?:\s+(?:but|however|although|whereas|while|yet|'
        r'because|since|so|therefore|nevertheless|nonetheless)\s+)', text)
    parts = [p.strip() for p in parts if len(p.strip()) > 5]
    return parts[:MAX_CLAUSES] if parts else [text[:500]]

class TaskDataset(Dataset):
    """
    Returns:
      input_ids / attention_mask  — joint [question SEP answer] encoding
      q_input_ids / q_attn_mask   — question-only encoding for gating
      clause_ids / clause_masks   — per-clause encodings [MAX_CLAUSES, CLAUSE_LEN]
      n_clauses                   — number of real (non-padded) clauses
    """
    def __init__(self, ta, tb, labels, max_len, task_id):
        self.ta,self.tb,self.labels,self.max_len,self.task_id = ta,tb,labels,max_len,task_id
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        # Joint encoding
        enc = tokenizer(self.ta[idx],self.tb[idx],truncation=True,
                        padding="max_length",max_length=self.max_len,return_tensors="pt")
        # Question-only
        qenc = tokenizer(self.ta[idx],truncation=True,
                         padding="max_length",max_length=Q_MAX_LEN,return_tensors="pt")
        # Clause encodings (evasion uses real clauses; stance uses single clause)
        clauses = split_into_clauses(self.tb[idx]) if self.task_id==0 else [self.tb[idx]]
        ids_l, masks_l = [], []
        for c in clauses[:MAX_CLAUSES]:
            ce = tokenizer(self.ta[idx],c,truncation=True,
                           padding="max_length",max_length=CLAUSE_LEN,return_tensors="pt")
            ids_l.append(ce["input_ids"].squeeze(0))
            masks_l.append(ce["attention_mask"].squeeze(0))
        n_real = len(ids_l)
        # Pad to MAX_CLAUSES
        while len(ids_l) < MAX_CLAUSES:
            ids_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
            masks_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "q_input_ids":     qenc["input_ids"].squeeze(0),
            "q_attention_mask":qenc["attention_mask"].squeeze(0),
            "clause_ids":      torch.stack(ids_l),
            "clause_masks":    torch.stack(masks_l),
            "n_clauses":       torch.tensor(n_real, dtype=torch.long),
            "labels":          torch.tensor(self.labels[idx], dtype=torch.long),
            "task_id":         torch.tensor(self.task_id, dtype=torch.long),
        }

def make_loader(ta,tb,y,tid,ml,shuffle):
    return DataLoader(TaskDataset(ta,tb,y,ml,tid),batch_size=BATCH_SIZE,
                      shuffle=shuffle,num_workers=2,pin_memory=True)

loaders = {
    "evasion_train": make_loader(Qetr,Aetr,yetr,0,MAX_LEN,True),
    "evasion_val":   make_loader(Qeva,Aeva,yeva,0,MAX_LEN,False),
    "evasion_test":  make_loader(Qete,Aete,yete,0,MAX_LEN,False),
    "stance_train":  make_loader(Qstr,Astr,ystr,1,STANCE_MAX_LEN,True),
    "stance_val":    make_loader(Qsva,Asva,ysva,1,STANCE_MAX_LEN,False),
    "stance_test":   make_loader(Qste,Aste,yste,1,STANCE_MAX_LEN,False),
}
for k,v in loaders.items():
    print(f"  {k:<20} {len(v.dataset):>6,} samples  {len(v):>5,} batches")
print("DataLoaders ready.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  evasion_train         2,758 samples    173 batches
  evasion_val             345 samples     22 batches
  evasion_test            345 samples     22 batches
  stance_train         17,224 samples  1,077 batches
  stance_val            2,193 samples    138 batches
  stance_test           2,157 samples    135 batches
DataLoaders ready.


In [7]:
def mean_pooling(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return torch.sum(h*m,1) / torch.clamp(m.sum(1),min=1e-9)

class TokenLevelGating(nn.Module):
    """Same target-aware token-level gating as V2."""
    def __init__(self, H, dr):
        super().__init__()
        self.Wa   = nn.Linear(H, H, bias=False)
        self.Wq   = nn.Linear(H, H, bias=True)
        self.drop = nn.Dropout(dr)
    def forward(self, h_seq, attention_mask, q_rep):
        gate = torch.sigmoid((self.Wa(h_seq) + self.Wq(q_rep).unsqueeze(1)) / 0.7)
        h_gated = self.drop(gate * h_seq)
        m = attention_mask.unsqueeze(-1).expand(h_gated.size()).float()
        return torch.sum(h_gated * m, 1) / torch.clamp(m.sum(1), min=1e-9)

class CrossAttention(nn.Module):
    """
    Scaled dot-product cross-attention (Augenstein et al. 2016).
    q_rep (question) attends over clause representations.
    Returns a question-weighted summary: which clauses matter most for the question.
    """
    def __init__(self, H):
        super().__init__()
        self.Wq = nn.Linear(H,H); self.Wk = nn.Linear(H,H); self.Wv = nn.Linear(H,H)
    def forward(self, q, clause_reps, n_clauses):
        # q: [B,H], clause_reps: [B,C,H]
        B, C, H = clause_reps.size()
        Q = self.Wq(q).unsqueeze(1)          # [B,1,H]
        K = self.Wk(clause_reps)              # [B,C,H]
        V = self.Wv(clause_reps)              # [B,C,H]
        scale = torch.sqrt(torch.tensor(H, dtype=torch.float32, device=q.device))
        scores = torch.bmm(Q, K.transpose(1,2)) / scale
        # mask padding clauses
        valid = torch.arange(C, device=q.device).unsqueeze(0) < n_clauses.unsqueeze(1)
        scores = scores.squeeze(1).masked_fill(~valid, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = weights / (weights.sum(-1, keepdim=True) + 1e-9)
        weights = F.dropout(weights, p=0.1, training=self.training)
        weights = weights.unsqueeze(1)      # [B,1,C]
        out = torch.bmm(weights, V).squeeze(1)                # [B,H]
        return out

class GraphReasoningLayer(nn.Module):
    """
    2-round message passing over clause nodes (Gong et al. 2024).
    Each clause aggregates mean of all other REAL clauses, then updates:
        LayerNorm( x + MLP(concat(x, neighbour_mean)) )
    Padding clauses are masked at every round.
    Propagates evasion/stance signals across the full answer.
    """
    def __init__(self, H, dr, rounds=2):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(H*2,H), nn.ReLU(), nn.Dropout(dr))
            for _ in range(rounds)])
        self.norm = nn.LayerNorm(H)
    def forward(self, x, n_clauses):
        B, C, H = x.size()
        valid = (torch.arange(C, device=x.device).unsqueeze(0) < n_clauses.unsqueeze(1)).float()
        for layer in self.layers:
            # mean of all OTHER real clause reps as neighbourhood signal
            sum_x    = (x * valid.unsqueeze(-1)).sum(1, keepdim=True)  # [B,1,H]
            denom = (n_clauses.unsqueeze(-1).unsqueeze(-1) - 1).clamp(min=1)
            nbr_mean = (sum_x - x * valid.unsqueeze(-1)) / denom
            x = self.norm(x + layer(torch.cat([x, nbr_mean], dim=-1)) * valid.unsqueeze(-1))
        return x

class ClauseAttentionPooling(nn.Module):
    """
    Attention-weighted pooling over graph-updated clause reps (Yang et al. 2016).
    Learns an importance scalar per clause; padding clauses masked to -inf.
    """
    def __init__(self, H, dr):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(H,H), nn.Tanh(), nn.Linear(H,1))
        self.norm  = nn.LayerNorm(H)
        self.drop  = nn.Dropout(dr)
    def forward(self, x, n_clauses):
        B, C, _ = x.size()
        valid = (torch.arange(C, device=x.device).unsqueeze(0) < n_clauses.unsqueeze(1))
        s = self.score(x).squeeze(-1).masked_fill(~valid, float("-inf"))
        w = F.softmax(s, dim=-1).unsqueeze(-1)
        return self.norm(self.drop((w * x).sum(1)))

class ClassHead(nn.Module):
    def __init__(self, H, n, dr):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dr), nn.Linear(H,256), nn.ReLU(),
                                 nn.Dropout(dr), nn.Linear(256,n))
    def forward(self, x): return self.net(x)

class MultiTaskDistilBERT_Advanced(nn.Module):
    """
    Variant 3 — Advanced multi-component architecture.

    Pipeline (evasion task):
      1. Shared DistilBERT encodes [question SEP answer] -> global mean-pool
      2. Shared DistilBERT encodes [question] -> q_rep
      3. Token-level gating: gate each answer token by q_rep -> gated_pool
      4. Per-clause encoding: DistilBERT encodes each clause -> clause_reps [B,C,H]
      5. Cross-attention: q_rep attends over clause_reps -> xattn_pool [B,H]
      6. Graph reasoning: 2-round message passing updates clause_reps
      7. Clause attention pooling: weighted sum of graph-updated clauses -> cl_pool
      8. Fusion: LayerNorm( gated_pool + xattn_pool + cl_pool ) -> ClassHead

    For stance task: clauses=[tweet] so clause pipeline degenerates gracefully.

    Improvements over V2:
      - Cross-attention finds which clauses are most relevant to the question
      - Graph reasoning propagates signals across clauses (catches distributed evasion)
      - Clause attention pooling weights clauses by importance
      - Fusion combines global + clause-aware representations
    """
    def __init__(self, ew, sw):
        super().__init__()
        self.encoder   = DistilBertModel.from_pretrained(BERT_MODEL)
        H = self.encoder.config.hidden_size
        # Shared components
        self.clause_proj = nn.Linear(H, H)
        self.cross_attn  = CrossAttention(H)
        self.graph       = GraphReasoningLayer(H, DROPOUT, rounds=2)
        self.cl_pool     = ClauseAttentionPooling(H, DROPOUT)
        self.fusion      = nn.LayerNorm(H)
        # Per-task gating (same as V2)
        self.evasion_gate = TokenLevelGating(H, DROPOUT)
        self.stance_gate  = TokenLevelGating(H, DROPOUT)
        # Per-task heads
        self.evasion_head    = ClassHead(H, len(EVASION_LABELS), DROPOUT)
        self.stance_head     = ClassHead(H, len(STANCE_LABELS),  DROPOUT)
        self.evasion_loss_fn = nn.CrossEntropyLoss(weight=ew)
        self.stance_loss_fn  = nn.CrossEntropyLoss(weight=sw)

    def _encode_clauses(self, clause_ids, clause_masks, B, C):

        ids = clause_ids.view(B*C, -1)
        masks = clause_masks.view(B*C, -1)
    
        valid = masks.sum(dim=1) > 0
        valid_ids = ids[valid]
        valid_masks = masks[valid]

        if valid_ids.size(0) == 0:
            return torch.zeros(B, C, self.encoder.config.hidden_size, device=clause_ids.device)
            
        valid_h = self.encoder(
            input_ids=valid_ids,
            attention_mask=valid_masks
        ).last_hidden_state
    
        pooled_valid = mean_pooling(valid_h, valid_masks)
    
        pool = torch.zeros(ids.size(0), pooled_valid.size(-1), device=ids.device)
        pool[valid] = pooled_valid
    
        pool = F.relu(self.clause_proj(pool))
    
        return pool.view(B, C, -1)



    def freeze_inactive_head(self, task):
        shared = [self.encoder, self.clause_proj, self.cross_attn,
                  self.graph, self.cl_pool, self.fusion]
        if task == 0:
            for m in [self.stance_head, self.stance_gate]:
                for p in m.parameters(): p.requires_grad=False
            for m in [self.evasion_head, self.evasion_gate]:
                for p in m.parameters(): p.requires_grad=True
        else:
            for m in [self.evasion_head, self.evasion_gate]:
                for p in m.parameters(): p.requires_grad=False
            for m in [self.stance_head, self.stance_gate]:
                for p in m.parameters(): p.requires_grad=True
        for m in shared:
            for p in m.parameters(): p.requires_grad=True

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad=True

    def forward(self, input_ids, attention_mask, task_id, labels=None,
                q_input_ids=None, q_attention_mask=None,
                clause_ids=None, clause_masks=None, n_clauses=None):

        B = input_ids.size(0)
        task = int(task_id[0].item())

        # 1. Encode joint [question SEP answer]
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        # 2. Encode question alone -> q_rep
        q_h   = self.encoder(input_ids=q_input_ids, attention_mask=q_attention_mask).last_hidden_state
        q_rep = F.normalize(mean_pooling(q_h, q_attention_mask), dim=-1)   # [B, H]

        # 3. Token-level gating (same as V2)
        gate_fn = self.evasion_gate if task==0 else self.stance_gate
        gated_pool = gate_fn(h, attention_mask, q_rep)   # [B, H]

        # 4-7. Clause hierarchy pipeline
        C = clause_ids.size(1)
        clause_reps = self._encode_clauses(clause_ids, clause_masks, B, C)   # [B,C,H]
        xattn_pool  = self.cross_attn(q_rep, clause_reps, n_clauses)          # [B,H]
        clause_reps = self.graph(clause_reps, n_clauses)                       # [B,C,H]
        cl_pool     = self.cl_pool(clause_reps, n_clauses)                     # [B,H]

        # 8. Fusion: sum of three complementary representations
        fused = self.fusion(F.dropout(gated_pool + 0.7*xattn_pool + 0.7*cl_pool,p=0.2, training=self.training))
        head_fn = self.evasion_head if task==0 else self.stance_head
        logits  = head_fn(fused)
        loss_fn = self.evasion_loss_fn if task==0 else self.stance_loss_fn
        loss    = loss_fn(logits, labels) if labels is not None else None
        return loss, logits

ew = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1,2]),y=yetr),dtype=torch.float).to(device)
sw = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1]),  y=ystr),dtype=torch.float).to(device)
print(f"Evasion weights: {ew.tolist()}")
print(f"Stance  weights: {sw.tolist()}")
model = MultiTaskDistilBERT_Advanced(ew, sw).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Arch: DistilBERT -> token-gate + CrossAttn + GraphReasoning + ClausePool -> Fusion -> ClassHead x2")

Evasion weights: [0.7462121248245239, 14.592592239379883, 0.6283891797065735]
Stance  weights: [1.0317479372024536, 0.9701475501060486]


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 74,437,894
Arch: DistilBERT -> token-gate + CrossAttn + GraphReasoning + ClausePool -> Fusion -> ClassHead x2


In [8]:
@torch.no_grad()
def evaluate(loader, label_names):
    model.eval(); model.unfreeze_all()
    all_preds, all_labels, total_loss = [], [], 0.0
    for batch in loader:
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device),
            batch["q_input_ids"].to(device),
            batch["q_attention_mask"].to(device),
            batch["clause_ids"].to(device),
            batch["clause_masks"].to(device),
            batch["n_clauses"].to(device))
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits,1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
    return {
        "loss":   total_loss/len(loader),
        "acc":    accuracy_score(all_labels,all_preds),
        "f1":     f1_score(all_labels,all_preds,average="macro",zero_division=0),
        "prec":   precision_score(all_labels,all_preds,average="macro",zero_division=0),
        "rec":    recall_score(all_labels,all_preds,average="macro",zero_division=0),
        "report": classification_report(all_labels,all_preds,target_names=label_names,zero_division=0),
        "cm":     pd.DataFrame(confusion_matrix(all_labels,all_preds,labels=list(range(len(label_names)))),
                    index=[f"True:{l}" for l in label_names],
                    columns=[f"Pred:{l}" for l in label_names]),
        "preds": all_preds, "labels": all_labels,
    }

def print_eval(m, title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(f"  Loss:{m['loss']:.4f}  Acc:{m['acc']*100:.2f}%  MacroF1:{m['f1']:.4f}  Prec:{m['prec']:.4f}  Rec:{m['rec']:.4f}")
    print("\n  Per-class Report:")
    for line in m["report"].strip().split("\n"): print(f"  {line}")
    print("\n  Confusion Matrix:")
    print(m["cm"].to_string())
    print("="*60)

print("Evaluation functions ready.")

Evaluation functions ready.


In [9]:
shared_params = (list(model.encoder.parameters())    +
                 list(model.cross_attn.parameters())  +
                 list(model.clause_proj.parameters()) +
                 list(model.graph.parameters())       +
                 list(model.cl_pool.parameters())     +
                 list(model.fusion.parameters()))

optimizer = AdamW([
    {"params": list(model.evasion_head.parameters())+list(model.evasion_gate.parameters()),
     "lr": LR_EVASION},
    {"params": list(model.stance_head.parameters())+list(model.stance_gate.parameters()),
     "lr": LR_STANCE},
    {"params": shared_params, "lr": LR_EVASION},
], weight_decay=0.01)

n_steps  = len(loaders["stance_train"]) * 3 * EPOCHS
n_warmup = int(WARMUP_RATIO * n_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, n_warmup, n_steps)
print(f"Optimizer ready. Steps: {n_steps}  Warmup: {n_warmup}")

Optimizer ready. Steps: 16155  Warmup: 1615


In [10]:
import itertools

best_avg_f1 = 0.0
history     = []

print("="*60)
print(f"  TRAINING — {VARIANT_NAME}")
print(f"  Epochs: {EPOCHS}  BERT frozen first {FREEZE_EPOCHS} epochs")
print(f"  LR evasion: {LR_EVASION}  LR stance: {LR_STANCE}  Batch: {BATCH_SIZE}")
print("="*60)

for p in model.encoder.parameters():
    p.requires_grad = False
print(f"BERT encoder FROZEN for epochs 1-{FREEZE_EPOCHS}.")

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:
        for p in model.encoder.parameters():
            p.requires_grad = True
        print(f"\n  Epoch {epoch}: BERT UNFROZEN — full fine-tuning begins.")

    model.train()
    run_e_loss = run_s_loss = 0.0
    n_e = n_s = 0

    e_cycle  = itertools.cycle(list(loaders["evasion_train"]))
    combined = []
    for s_batch in loaders["stance_train"]:
        combined.append(next(e_cycle))
        combined.append(next(e_cycle))
        combined.append(s_batch)

    total = len(combined)
    print(f"\nEpoch {epoch}/{EPOCHS} — {total} steps")

    for step, batch in enumerate(combined, 1):
        task = int(batch["task_id"][0].item())
        model.freeze_inactive_head(task)
        optimizer.zero_grad()
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device),
            batch["q_input_ids"].to(device),
            batch["q_attention_mask"].to(device),
            batch["clause_ids"].to(device),
            batch["clause_masks"].to(device),
            batch["n_clauses"].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        if task == 0: run_e_loss += loss.item(); n_e += 1
        else:         run_s_loss += loss.item(); n_s += 1
        done = int(30 * step / total)
        print(f"\r  [{'='*done}{'-'*(30-done)}] {step}/{total}", end="", flush=True)

    avg_e = run_e_loss / n_e if n_e else 0
    avg_s = run_s_loss / n_s if n_s else 0
    print()

    print("  Validating...")
    e_val  = evaluate(loaders["evasion_val"], EVASION_LABELS)
    s_val  = evaluate(loaders["stance_val"],  STANCE_LABELS)
    avg_f1 = (e_val["f1"] + s_val["f1"]) / 2
    is_best = avg_f1 > best_avg_f1

    print(f"\n  Epoch {epoch}/{EPOCHS} Summary")
    print(f"  {'-'*52}")
    print(f"  {'Metric':<24} {'Evasion':>10} {'Stance':>10}")
    print(f"  {'-'*52}")
    print(f"  {'Train Loss':<24} {avg_e:>10.4f} {avg_s:>10.4f}")
    print(f"  {'Val Loss':<24} {e_val['loss']:>10.4f} {s_val['loss']:>10.4f}")
    print(f"  {'Accuracy':<24} {e_val['acc']*100:>9.2f}% {s_val['acc']*100:>9.2f}%")
    print(f"  {'Macro F1':<24} {e_val['f1']:>10.4f} {s_val['f1']:>10.4f}")
    print(f"  {'Precision':<24} {e_val['prec']:>10.4f} {s_val['prec']:>10.4f}")
    print(f"  {'Recall':<24} {e_val['rec']:>10.4f} {s_val['rec']:>10.4f}")
    print(f"  {'-'*52}")
    print(f"  Avg Macro F1: {avg_f1:.4f}  {'<- BEST' if is_best else ''}")

    if is_best:
        best_avg_f1 = avg_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  Saved best model -> {BEST_MODEL_PATH}")

    history.append({"epoch":epoch,"e_f1":e_val["f1"],"s_f1":s_val["f1"],"avg_f1":avg_f1})

print("\n" + "="*60)
print(f"  Training complete.  Best Avg Macro F1: {best_avg_f1:.4f}")
print("="*60)

  TRAINING — Variant 3 — Advanced (CrossAttn + GraphReasoning + ClauseHierarchy)
  Epochs: 5  BERT frozen first 2 epochs
  LR evasion: 1e-05  LR stance: 2e-05  Batch: 16
BERT encoder FROZEN for epochs 1-2.

Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  Validating...

  Epoch 1/5 Summary
  ----------------------------------------------------
  Metric                      Evasion     Stance
  ----------------------------------------------------
  Train Loss                   0.9627     0.6215
  Val Loss                     2.1544     0.5607
  Accuracy                     51.30%     73.42%
  Macro F1                     0.3809     0.7330
  Precision                    0.3774     0.7343
  Recall                       0.3883     0.7327
  ----------------------------------------------------
  Avg Macro F1: 0.5569  <- BEST
  Saved best model -> /kaggle/working/best_v3_advanced.pt

Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  Validating...

  Ep

In [11]:
print(f"Loading best model: {BEST_MODEL_PATH}")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.unfreeze_all()
print("Model loaded.\n")

e_test = evaluate(loaders["evasion_test"], EVASION_LABELS)
s_test = evaluate(loaders["stance_test"],  STANCE_LABELS)

print_eval(e_test, f"QA EVASION — Test  [{VARIANT_NAME}]")
print_eval(s_test, f"STANCE    — Test  [{VARIANT_NAME}]")

pd.DataFrame({"question":Qete,"answer":Aete,
    "true":[EVASION_LABELS[l] for l in e_test["labels"]],
    "pred":[EVASION_LABELS[p] for p in e_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,f"results_{SAVE_TAG}_evasion.csv"),index=False)

pd.DataFrame({"target":Qste,"tweet":Aste,
    "true":[STANCE_LABELS[l] for l in s_test["labels"]],
    "pred":[STANCE_LABELS[p] for p in s_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,f"results_{SAVE_TAG}_stance.csv"),index=False)

print(f"\n{'='*60}")
print(f"  {VARIANT_NAME} — Final Results")
print(f"  Evasion Macro F1 : {e_test['f1']:.4f}")
print(f"  Stance  Macro F1 : {s_test['f1']:.4f}")
print(f"  Avg Macro F1     : {(e_test['f1']+s_test['f1'])/2:.4f}")
print(f"{'='*60}")
print("Files saved to OUTPUT_DIR.")

Loading best model: /kaggle/working/best_v3_advanced.pt
Model loaded.


  QA EVASION — Test  [Variant 3 — Advanced (CrossAttn + GraphReasoning + ClauseHierarchy)]
  Loss:3.5217  Acc:62.61%  MacroF1:0.4692  Prec:0.4763  Rec:0.4703

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasive       0.58      0.70      0.63       154
  Partially Evasive       0.14      0.12      0.13         8
            Evasive       0.71      0.58      0.64       183
  
           accuracy                           0.63       345
          macro avg       0.48      0.47      0.47       345
       weighted avg       0.64      0.63      0.63       345

  Confusion Matrix:
                        Pred:Non-Evasive  Pred:Partially Evasive  Pred:Evasive
True:Non-Evasive                     108                       4            42
True:Partially Evasive                 5                       1             2
True:Evasive                          74                       2           10

In [12]:
print("All done.")

All done.
